# 负载均衡教程

本教程详细介绍负载均衡的原理和实现，包括：

1. **负载均衡策略**: 轮询、加权、最少连接等
2. **健康检查**: 服务器状态监控
3. **限流器**: 令牌桶算法
4. **熔断器**: 故障隔离
5. **完整负载均衡器**: 生产级实现

---

## 负载均衡架构

```
                    ┌─────────────────────────────────┐
                    │         负载均衡器               │
                    │    (Nginx / HAProxy / K8s)      │
                    └─────────────┬───────────────────┘
                                  │
            ┌─────────────────────┼─────────────────────┐
            │                     │                     │
            ▼                     ▼                     ▼
    ┌───────────────┐     ┌───────────────┐     ┌───────────────┐
    │   服务实例 1   │     │   服务实例 2   │     │   服务实例 3   │
    │  (FastAPI)    │     │  (FastAPI)    │     │  (FastAPI)    │
    └───────────────┘     └───────────────┘     └───────────────┘
```

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

import asyncio
import time
import numpy as np

# 导入负载均衡模块
from load_balancer import (
    HTTPX_AVAILABLE,
    LoadBalanceStrategy,
    Server,
    RoundRobinStrategy,
    WeightedRoundRobinStrategy,
    LeastConnectionsStrategy,
    IPHashStrategy,
    RandomStrategy,
    ResponseTimeStrategy,
    RateLimiter,
    CircuitBreaker,
    CircuitBreakerState,
    LoadBalancer,
    create_load_balancer,
)

print(f"httpx 可用: {HTTPX_AVAILABLE}")
if not HTTPX_AVAILABLE:
    print("安装命令: pip install httpx")

## 1. 服务器节点

Server 类表示后端服务器节点，包含健康状态、连接数、响应时间等信息。

In [ ]:
# 创建服务器节点
servers = [
    Server(url="http://server1:8000", weight=3),
    Server(url="http://server2:8000", weight=2),
    Server(url="http://server3:8000", weight=1),
]

print("服务器节点:")
for s in servers:
    print(f"  {s.url}: 权重={s.weight}, 健康={s.healthy}")

In [ ]:
# 模拟记录响应
server = servers[0]

# 记录一些响应
for i in range(20):
    latency = np.random.exponential(10)  # 模拟延迟
    success = np.random.random() > 0.1   # 90% 成功率
    server.record_response(latency, success)

print(f"服务器统计:")
print(f"  平均响应时间: {server.avg_response_time:.2f}ms")
print(f"  成功率: {server.success_rate:.1%}")
print(f"  成功次数: {server.success_count}")
print(f"  失败次数: {server.failure_count}")

## 2. 负载均衡策略

### 2.1 轮询策略 (Round Robin)

按顺序依次选择服务器。

In [ ]:
# 轮询策略
rr_strategy = RoundRobinStrategy()

servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

print("轮询策略选择顺序:")
for i in range(9):
    selected = rr_strategy.select(servers)
    print(f"  请求 {i+1}: {selected.url}")

### 2.2 加权轮询策略 (Weighted Round Robin)

根据权重分配请求，权重高的服务器处理更多请求。

In [ ]:
# 加权轮询策略
wrr_strategy = WeightedRoundRobinStrategy()

weighted_servers = [
    Server(url="http://server1:8000", weight=3),  # 50%
    Server(url="http://server2:8000", weight=2),  # 33%
    Server(url="http://server3:8000", weight=1),  # 17%
]

# 统计分布
counts = {s.url: 0 for s in weighted_servers}
for _ in range(600):
    selected = wrr_strategy.select(weighted_servers)
    counts[selected.url] += 1

print("加权轮询分布 (权重 3:2:1):")
for url, count in counts.items():
    print(f"  {url}: {count} 次 ({count/600:.1%})")

### 2.3 最少连接策略 (Least Connections)

选择当前连接数最少的服务器。

In [ ]:
# 最少连接策略
lc_strategy = LeastConnectionsStrategy()

lc_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 设置不同的连接数
lc_servers[0].connections = 10
lc_servers[1].connections = 5
lc_servers[2].connections = 8

print("服务器连接数:")
for s in lc_servers:
    print(f"  {s.url}: {s.connections} 连接")

selected = lc_strategy.select(lc_servers)
print(f"\n选择: {selected.url} (连接数最少)")

### 2.4 IP 哈希策略 (IP Hash)

相同 IP 的请求总是路由到相同服务器，实现会话保持。

In [ ]:
# IP 哈希策略
ip_strategy = IPHashStrategy()

ip_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 测试相同 IP 的一致性
test_ips = ["192.168.1.100", "192.168.1.101", "10.0.0.1"]

print("IP 哈希一致性测试:")
for ip in test_ips:
    results = set()
    for _ in range(10):
        selected = ip_strategy.select(ip_servers, client_ip=ip)
        results.add(selected.url)
    print(f"  IP {ip}: 总是路由到 {list(results)[0]}")

### 2.5 响应时间策略

选择平均响应时间最短的服务器。

In [ ]:
# 响应时间策略
rt_strategy = ResponseTimeStrategy()

rt_servers = [
    Server(url="http://server1:8000"),
    Server(url="http://server2:8000"),
    Server(url="http://server3:8000"),
]

# 模拟不同的响应时间
for _ in range(10):
    rt_servers[0].record_response(100.0, True)  # 慢
    rt_servers[1].record_response(20.0, True)   # 快
    rt_servers[2].record_response(50.0, True)   # 中等

print("服务器响应时间:")
for s in rt_servers:
    print(f"  {s.url}: {s.avg_response_time:.1f}ms")

selected = rt_strategy.select(rt_servers)
print(f"\n选择: {selected.url} (响应最快)")

## 3. 限流器 (Rate Limiter)

使用令牌桶算法实现限流，防止服务过载。

```
令牌桶算法:
┌─────────────────┐
│  ○ ○ ○ ○ ○ ○    │  ← 令牌以固定速率生成
│  ○ ○ ○ ○        │
│  ○ ○            │  桶容量限制最大令牌数
└────────┬────────┘
         │
         ▼
    请求消耗令牌
```

In [ ]:
# 创建限流器: 每秒 10 个令牌，桶容量 5
limiter = RateLimiter(rate=10.0, capacity=5)

print("限流器配置:")
print(f"  速率: {limiter.rate} 令牌/秒")
print(f"  容量: {limiter.capacity} 令牌")

In [ ]:
# 测试限流
async def test_rate_limiter():
    limiter = RateLimiter(rate=10.0, capacity=5)
    
    print("快速请求测试 (应该有些被拒绝):")
    results = []
    for i in range(10):
        success = await limiter.acquire()
        results.append(success)
        print(f"  请求 {i+1}: {'通过' if success else '拒绝'}")
    
    print(f"\n通过: {sum(results)}, 拒绝: {len(results) - sum(results)}")
    
    # 等待令牌补充
    print("\n等待 0.5 秒后再次请求...")
    await asyncio.sleep(0.5)
    
    success = await limiter.acquire()
    print(f"请求: {'通过' if success else '拒绝'}")

await test_rate_limiter()

## 4. 熔断器 (Circuit Breaker)

熔断器用于故障隔离，防止级联故障。

```
熔断器状态:

CLOSED (正常)  ──失败达到阈值──→  OPEN (熔断)
    ↑                                │
    │                          超时后
    │                                ↓
    └──────成功──────  HALF_OPEN (半开)
                           │
                      失败则重新 OPEN
```

In [ ]:
# 创建熔断器
breaker = CircuitBreaker(
    failure_threshold=3,      # 3 次失败触发熔断
    recovery_timeout=1.0,     # 1 秒后尝试恢复
    half_open_requests=2      # 半开状态允许 2 个请求
)

print("熔断器配置:")
print(f"  失败阈值: {breaker.failure_threshold}")
print(f"  恢复超时: {breaker.recovery_timeout}s")
print(f"  半开请求数: {breaker.half_open_requests}")
print(f"  当前状态: {breaker.state.value}")

In [ ]:
# 测试熔断器状态转换
async def test_circuit_breaker():
    breaker = CircuitBreaker(
        failure_threshold=3,
        recovery_timeout=0.5,
        half_open_requests=2
    )
    
    print("1. 初始状态 (CLOSED):")
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {await breaker.can_execute()}")
    
    print("\n2. 记录 3 次失败:")
    for i in range(3):
        await breaker.record_failure()
        print(f"   失败 {i+1}: 状态={breaker.state.value}")
    
    print("\n3. 熔断状态 (OPEN):")
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {await breaker.can_execute()}")
    
    print("\n4. 等待恢复超时...")
    await asyncio.sleep(0.6)
    
    print("\n5. 半开状态 (HALF_OPEN):")
    can_exec = await breaker.can_execute()
    print(f"   状态: {breaker.state.value}")
    print(f"   可执行: {can_exec}")
    
    print("\n6. 记录成功，恢复正常:")
    await breaker.record_success()
    await breaker.record_success()
    print(f"   状态: {breaker.state.value}")

await test_circuit_breaker()

## 5. 完整负载均衡器

LoadBalancer 类整合了所有功能：负载均衡策略、健康检查、限流、熔断。

In [ ]:
# 创建负载均衡器
balancer = create_load_balancer(
    servers=[
        "http://server1:8000",
        "http://server2:8000",
        "http://server3:8000",
    ],
    strategy="round_robin",
    weights=[3, 2, 1],
    enable_circuit_breaker=True,
    rate_limit=100.0,  # 每秒 100 请求
)

print("负载均衡器配置:")
print(f"  策略: {balancer.strategy_type.value}")
print(f"  服务器数: {len(balancer.servers)}")
print(f"  熔断器: {balancer.enable_circuit_breaker}")
print(f"  限流: {balancer.rate_limiter is not None}")

In [ ]:
# 获取统计信息
stats = balancer.get_stats()

print("负载均衡器统计:")
print(f"  策略: {stats['strategy']}")
print(f"\n服务器状态:")
for server in stats['servers']:
    print(f"  {server['url']}:")
    print(f"    健康: {server['healthy']}")
    print(f"    权重: {server['weight']}")
    print(f"    连接数: {server['connections']}")

## 6. 策略对比

In [ ]:
# 策略对比表
print("负载均衡策略对比:")
print("=" * 70)
print(f"{'策略':<20} {'优点':<25} {'适用场景'}")
print("-" * 70)

strategies = [
    ("轮询", "简单公平", "服务器性能相近"),
    ("加权轮询", "考虑服务器能力", "服务器性能不同"),
    ("最少连接", "动态负载均衡", "长连接场景"),
    ("IP 哈希", "会话保持", "有状态服务"),
    ("随机", "简单高效", "无状态服务"),
    ("响应时间", "性能优先", "延迟敏感场景"),
]

for name, pros, scenario in strategies:
    print(f"{name:<20} {pros:<25} {scenario}")

## 7. 生产部署示例

In [ ]:
print("NGINX 负载均衡配置示例:")
print("""
upstream model_servers {
    least_conn;  # 最少连接策略
    server model-server-1:8000 weight=3;
    server model-server-2:8000 weight=2;
    server model-server-3:8000 weight=1;
    keepalive 32;  # 保持连接
}

server {
    listen 80;

    location /predict {
        proxy_pass http://model_servers;
        proxy_http_version 1.1;
        proxy_set_header Connection "";
        proxy_connect_timeout 10s;
        proxy_read_timeout 60s;
    }

    location /health {
        return 200 'OK';
    }
}
""")

In [ ]:
print("Kubernetes Service 配置示例:")
print("""
apiVersion: v1
kind: Service
metadata:
  name: model-server
spec:
  selector:
    app: model-server
  ports:
  - port: 80
    targetPort: 8000
  type: LoadBalancer
  sessionAffinity: ClientIP  # IP 哈希
  sessionAffinityConfig:
    clientIP:
      timeoutSeconds: 3600
""")

## 总结

本教程介绍了负载均衡的核心概念：

1. **负载均衡策略**: 轮询、加权、最少连接、IP 哈希、响应时间
2. **限流器**: 令牌桶算法防止过载
3. **熔断器**: 故障隔离防止级联故障
4. **健康检查**: 自动检测服务器状态

### 最佳实践

- 根据业务场景选择合适的负载均衡策略
- 启用健康检查自动剔除故障节点
- 使用熔断器防止级联故障
- 配置限流保护后端服务
- 监控负载均衡器指标